# **Custom Functions and Variables**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from scipy.stats import spearmanr, pearsonr

In [ ]:
def read_json(json_path):
  with open(json_path, 'r') as f:
    return json.load(f)

In [ ]:
def num_to_word(num):
    if num <= -0.5:
        return 'Low'
    elif num >= 0.5:
        return 'High'
    else:
        return "Neutral"

In [ ]:
def within1(y_true, y_pred):
    yt = np.rint(np.clip(y_true, -3, 3))
    yp = np.rint(np.clip(y_pred, -3, 3))
    return float(np.mean(np.abs(yt - yp) <= 1))

In [ ]:
def return_stats(y_pred, y_true):
  """
  Print evaluation metrics for a classification model.

  This function computes and prints the following metrics, rounded to two
  decimal places and separated by tabs:
  - Accuracy
  - Macro-averaged F1 score
  - Within-1 accuracy (predictions within ±1 of the true label)
  - Macro-averaged precision
  - Macro-averaged recall
  """
  print(round(accuracy_score(y_pred, y_true),2), '\t', round(f1_score(y_pred, y_true, average='macro'), 2), '\t', round(within1(y_pred, y_true), 2), '\t', round(precision_score(y_pred, y_true, average='macro'), 2), '\t', round(recall_score(y_pred, y_true, average='macro'), 2))

# **Dummy Classifier**

In [ ]:
from sklearn.dummy import DummyClassifier

In [ ]:
def coarsen_ww(v):
  if v >= 2.5:
    return 3
  elif v >= 1.5:
    return 2
  elif v >= 0.5:
    return 1
  elif v >= -0.5:
    return 0
  elif v >= -1.5:
    return -1
  elif v >= -2.5:
    return -2
  else:
    return -3

In [ ]:
X_train = X_train[['Text', 'Target', 'Dimension', 'Score_mean']]

X_train['Score_coarse'] = X_train['Score_mean'].apply(lambda x: -1 if x <= -0.5 else 1 if x >= 0.5 else 0)

# Trust
X_train_trust_temp = X_train[X_train['Dimension'] == 'Trust'].copy()
y_train_trust = X_train_trust_temp['Score_coarse']
X_train_trust = X_train_trust_temp.drop(columns=['Score_coarse', 'Score_mean', 'Dimension'])

# Sociability
X_train_soc_temp = X_train[X_train['Dimension'] == 'Sociability'].copy()
y_train_soc = X_train_soc_temp['Score_coarse']
X_train_soc = X_train_soc_temp.drop(columns=['Score_mean', 'Score_coarse', 'Dimension'])

# Competence
X_train_comp_temp = X_train[X_train['Dimension'] == 'Competence'].copy()
y_train_comp = X_train_comp_temp['Score_coarse']
X_train_comp = X_train_comp_temp.drop(columns=['Score_mean', 'Score_coarse',  'Dimension'])

In [ ]:
clf_trust = DummyClassifier(strategy="most_frequent")
clf_trust.fit(X_train_trust, y_train_trust)
y_pred_trust = clf_trust.predict(X_test[['Text', 'Target']])

In [ ]:
clf_soc = DummyClassifier(strategy="most_frequent")
clf_soc.fit(X_train_soc, y_train_soc)
y_pred_soc = clf_soc.predict(X_test[['Text', 'Target']])

In [ ]:
clf_comp = DummyClassifier(strategy="most_frequent")
clf_comp.fit(X_train_comp, y_train_comp)
y_pred_comp = clf_comp.predict(X_test[['Text', 'Target']])

In [ ]:
X_test['Competence_ww'] = X_test['Competence_mean'].copy().apply(coarsen_ww)
X_test['Sociability_ww'] = X_test['Sociability_mean'].copy().apply(coarsen_ww)
X_test['Trust_ww'] = X_test['Trust_mean'].copy().apply(coarsen_ww)

In [ ]:
return_stats(X_test['Competence_ww'], y_pred_comp)

0.24 	 0.09 	 0.6 	 0.06 	 0.24


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
return_stats(X_test['Sociability_ww'], y_pred_soc)

0.26 	 0.11 	 0.67 	 0.07 	 0.26


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
return_stats(X_test['Trust_ww'], y_pred_trust)

0.22 	 0.08 	 0.58 	 0.05 	 0.22


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# **TF-IDF + Logistic Regression**

In [ ]:
def within1(y_true, y_pred):
    yt = np.rint(np.clip(y_true, -3, 3))
    yp = np.rint(np.clip(y_pred, -3, 3))
    return float(np.mean(np.abs(yt - yp) <= 1))

In [ ]:
from tqdm import tqdm
import numpy as np

In [ ]:
# TF-IDF + Logistic Regression baseline with grouped splits and paired inputs
# One model per Dimension, paired (Text, Target), and 7-class labels {-3..3}.

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# ---------- config ----------
score_col = "Score_mean"   # Dependent variable
max_features = 200_000
ngram_range = (1, 3)       # unigrams + bigrams as a baseline
min_df = 2                 # ignore rare words
C_grid = [2.0, 4.0, 6]
penalty = "l2"
solver = "saga"            # for multinomial and large sparse problems
max_iter = 5000
class_weight = "balanced"

In [ ]:
def grouped_splits(df: pd.DataFrame, test_size=0.2, val_size=0.2, seed=13):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss.split(df, groups=df["Text"]))
    trainval = df.iloc[trainval_idx].reset_index(drop=True)
    test     = df.iloc[test_idx].reset_index(drop=True)

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx, val_idx = next(gss2.split(trainval, groups=trainval["Text"]))
    train = trainval.iloc[train_idx].reset_index(drop=True)
    val   = trainval.iloc[val_idx].reset_index(drop=True)
    return train, val, test

def to_seven_classes(series: pd.Series) -> pd.Series:
    x = series.astype(float).values

    vals = np.select(
        [
            x >= 2.5,
            x >= 1.5,
            x >= 0.5,
            x >= -0.5,
            x >= -1.5,
            x >= -2.5
        ],
        [3, 2, 1, 0, -1, -2],
        default=-3
    )

    return pd.Series(vals, index=series.index)

def join_pair(a: pd.Series, b: pd.Series):
    # Pair input like your encode_pair, but for bag-of-words:
    # Keep a visible separator so bigrams don’t straddle the boundary.
    return (a.astype(str) + " [SEP] " + b.astype(str)).fillna("")

def fit_vectorizer(train_texts, *, ngram_range=ngram_range, min_df=min_df):
    vec = TfidfVectorizer(
        # max_features=max_features,
        ngram_range=ngram_range,
        min_df=min_df,
        lowercase=True,
        strip_accents="unicode",
        sublinear_tf=True
    )
    X = vec.fit_transform(train_texts)
    return vec, X

def evaluate_classifier(y_true, y_pred, labels_sorted):
    acc  = accuracy_score(y_true, y_pred)
    f1m  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1w  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    report = classification_report(y_true, y_pred, labels=labels_sorted, zero_division=0, digits=4)
    cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)
    return {
        "accuracy": float(acc),
        "f1_macro": float(f1m),
        "f1_weighted": float(f1w),
        "report": report,
        "confusion_matrix": cm
    }

def train_logreg_one_dim(df_all: pd.DataFrame, dim_name: str):
    # 1) filter and labelify
    df = df_all[df_all["Dimension"].str.strip() == dim_name].copy()
    df = df.dropna(subset=[score_col, "Text", "Target"])
    df["score_cont"] = df[score_col].astype(float).clip(-3, 3)
    df["label"] = to_seven_classes(df["score_cont"])   # {-3..3}

    # 2) splits (group by Text)
    train_df, val_df, test_df = grouped_splits(df, test_size=0.2, val_size=0.2, seed=13)

    # 3) build paired inputs
    train_texts = join_pair(train_df["Text"], train_df["Target"])
    val_texts   = join_pair(val_df["Text"],   val_df["Target"])
    test_texts  = join_pair(test_df["Text"],  test_df["Target"])

    # 4) TF-IDF fit on train only, transform all
    vec, X_train = fit_vectorizer(train_texts)
    X_val  = vec.transform(val_texts)
    X_test = vec.transform(test_texts)

    y_train = train_df["label"].values
    y_val   = val_df["label"].values
    y_test  = test_df["label"].values

    # Keep labels sorted for stable reporting
    labels_sorted = np.array(sorted(np.unique(np.concatenate([y_train, y_val, y_test]))))

    # 5) micro hyperparam search on C using validation macro-F1
    best = {"score": -np.inf, "C": None, "clf": None}
    for C in C_grid:
        clf = LogisticRegression(
            C=C,
            penalty=penalty,
            solver=solver,
            max_iter=max_iter,
            class_weight=class_weight,
            n_jobs=-1,
            # verbose=1
            # multi_class="multinomial"  # 7 classes
        )
        clf.fit(X_train, y_train)
        pred_val = clf.predict(X_val)
        f1m = f1_score(y_val, pred_val, average="macro", zero_division=0)
        if f1m > best["score"]:
            best.update({"score": f1m, "C": C, "clf": clf})

    # 6) optionally refit on train+val with best C
    X_trainval = np.vstack([X_train.toarray(), X_val.toarray()]) if hasattr(X_train, "toarray") else [X_train, X_val]
    if not isinstance(X_trainval, np.ndarray):
        from scipy.sparse import vstack
        X_trainval = vstack([X_train, X_val])
    y_trainval = np.concatenate([y_train, y_val])

    final_clf = LogisticRegression(
        C=best["C"],
        penalty=penalty,
        solver=solver,
        max_iter=max_iter,
        class_weight=class_weight,
        n_jobs=-1,
        # verbose=1
        # multi_class="multinomial"
    ).fit(X_trainval, y_trainval)

    # 7) evaluation
    pred_test = final_clf.predict(X_test)
    metrics = evaluate_classifier(y_test, pred_test, labels_sorted=labels_sorted)

    return {
        "vectorizer": vec,
        "classifier": final_clf,
        "best_C": best["C"],
        "val_f1_macro": float(best["score"]),
        "test_metrics": metrics,
        "y_true": y_test,
        "y_pred": pred_test,
        "test_df": test_df
    }

In [ ]:
results_clf = {}
for dim in ['Trust', 'Sociability', 'Competence']:
    results_clf[dim] = train_logreg_one_dim(concated_final_agg[["Text","Target","Dimension",score_col]], dim)

In [ ]:
print(results_clf['Trust']['test_metrics']['report'])

              precision    recall  f1-score   support

          -3     0.1970    0.3333    0.2476        39
          -2     0.3294    0.3636    0.3457        77
          -1     0.3488    0.2143    0.2655        70
           0     0.2895    0.2558    0.2716        43
           1     0.2273    0.3333    0.2703        30
           2     0.4118    0.2917    0.3415        48
           3     0.3529    0.3000    0.3243        20

    accuracy                         0.2966       327
   macro avg     0.3081    0.2989    0.2952       327
weighted avg     0.3167    0.2966    0.2982       327



In [ ]:
print(results_clf['Sociability']['test_metrics']['report'])

              precision    recall  f1-score   support

          -3     0.2500    0.3529    0.2927        17
          -2     0.3947    0.3333    0.3614        90
          -1     0.3494    0.3671    0.3580        79
           0     0.2778    0.3000    0.2885        50
           1     0.2000    0.2051    0.2025        39
           2     0.3684    0.3043    0.3333        46
           3     0.0000    0.0000    0.0000         6

    accuracy                         0.3119       327
   macro avg     0.2629    0.2661    0.2624       327
weighted avg     0.3242    0.3119    0.3163       327



In [ ]:
print(results_clf['Competence']['test_metrics']['report'])

              precision    recall  f1-score   support

          -3     0.0435    0.4000    0.0784         5
          -2     0.3200    0.1818    0.2319        44
          -1     0.1778    0.2133    0.1939        75
           0     0.2769    0.2195    0.2449        82
           1     0.2326    0.1449    0.1786        69
           2     0.1860    0.1739    0.1798        46
           3     0.0000    0.0000    0.0000         6

    accuracy                         0.1896       327
   macro avg     0.1767    0.1905    0.1582       327
weighted avg     0.2292    0.1896    0.2013       327



In [ ]:
import pprint

for item in results_clf.keys():
    print(item)
    print('Accuracy\t', round(results_clf[item]['test_metrics']['accuracy'], 3))
    print('F1 macro\t', round(results_clf[item]['test_metrics']['f1_macro'], 3))
    print('F1 weighted\t', round(results_clf[item]['test_metrics']['f1_weighted'], 3))
    print('Best C\t\t', results_clf[item]['best_C'])
    print()

Trust
Accuracy	 0.297
F1 macro	 0.295
F1 weighted	 0.298
Best C		 2.0

Sociability
Accuracy	 0.312
F1 macro	 0.262
F1 weighted	 0.316
Best C		 2.0

Competence
Accuracy	 0.19
F1 macro	 0.158
F1 weighted	 0.201
Best C		 6



In [ ]:
print('Within-1-accuracy:')
print('Trust:\t\t', round(within1(results_clf['Trust']['y_true'], results_clf['Trust']['y_pred']), 3))
print('Sociability:\t', round(within1(results_clf['Sociability']['y_true'], results_clf['Sociability']['y_pred']), 3))
print('Competence:\t', round(within1(results_clf['Competence']['y_true'], results_clf['Competence']['y_pred']), 3))

Within-1-accuracy:
Trust:		 0.651
Sociability:	 0.703
Competence:	 0.615


In [ ]:
from scipy.stats import spearmanr

In [ ]:
print('Spearman correlation:')
print('Trust:\t\t', round(spearmanr(results_clf['Trust']['y_true'], results_clf['Trust']['y_pred'])[0], 3))
print('Sociability:\t', round(spearmanr(results_clf['Sociability']['y_true'], results_clf['Sociability']['y_pred'])[0], 3))
print('Competence:\t', round(spearmanr(results_clf['Competence']['y_true'], results_clf['Competence']['y_pred'])[0], 3))

Spearman correlation:
Trust:		 0.441
Sociability:	 0.43
Competence:	 0.351


In [ ]:
from sklearn.metrics import root_mean_squared_error

print('RMSE:')
print('Trust:\t\t', round(float(root_mean_squared_error(results_clf['Trust']['y_true'], results_clf['Trust']['y_pred'])), 3))
print('Sociability:\t', round(float(root_mean_squared_error(results_clf['Sociability']['y_true'], results_clf['Sociability']['y_pred'])), 3))
print('Competence:\t', round(float(root_mean_squared_error(results_clf['Competence']['y_true'], results_clf['Competence']['y_pred'])), 3))

RMSE:
Trust:		 1.897
Sociability:	 1.657
Competence:	 1.769


In [ ]:
from sklearn.metrics import mean_absolute_error

print('MAE:')
print('Trust:\t\t', round(float(mean_absolute_error(results_clf['Trust']['y_true'], results_clf['Trust']['y_pred'])), 3))
print('Sociability:\t', round(float(mean_absolute_error(results_clf['Sociability']['y_true'], results_clf['Sociability']['y_pred'])), 3))
print('Competence:\t', round(float(mean_absolute_error(results_clf['Competence']['y_true'], results_clf['Competence']['y_pred'])), 3))

MAE:
Trust:		 1.355
Sociability:	 1.18
Competence:	 1.404


In [ ]:
from sklearn.metrics import mean_absolute_error

print('MAE:')
print('Trust:\t\t', round(float(mean_absolute_error(results_clf['Trust']['y_true'], results_clf['Trust']['y_pred'])), 3))
print('Sociability:\t', round(float(mean_absolute_error(results_clf['Sociability']['y_true'], results_clf['Sociability']['y_pred'])), 3))
print('Competence:\t', round(float(mean_absolute_error(results_clf['Competence']['y_true'], results_clf['Competence']['y_pred'])), 3))

MAE:
Trust:		 1.355
Sociability:	 1.18
Competence:	 1.404


# **BERT Classifiers**

## Imports

In [ ]:
!pip install evaluate
from dataclasses import dataclass
from typing import Dict, List
import numpy as np
import pandas as pd
import torch
import evaluate

from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from datasets import Dataset

clear_output()

In [ ]:
model_name = "vinai/bertweet-base"
tok = AutoTokenizer.from_pretrained(model_name, use_fast=False, normalization=True)

dim_names = ["Trust", "Sociability", "Competence"]
label_to_num = {"Low": 0, "Neutral": 1, "High": 2}
num_to_label = {v: k for k, v in label_to_num.items()}

clear_output()

## Tokenization and data split

In [ ]:
def encode_pair(df: pd.DataFrame, tokenizer, max_length=256):
    enc = tokenizer(
        df["Text"].tolist(),
        text_pair=df["Target"].tolist(),
        truncation=True,
        padding="max_length",
        max_length= max_length
    )
    enc["labels"] = df["label_id"].tolist()
    return enc

In [ ]:
def grouped_splits(df: pd.DataFrame, test_size=0.2, val_size=0.2, seed=13):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss.split(df, groups=df["Text"]))
    trainval = df.iloc[trainval_idx].reset_index(drop=True)
    test = df.iloc[test_idx].reset_index(drop=True)

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx, val_idx = next(gss2.split(trainval, groups=trainval["Text"]))
    train = trainval.iloc[train_idx].reset_index(drop=True)
    val   = trainval.iloc[val_idx].reset_index(drop=True)
    return train, val, test

In [ ]:
train_df_class, val_df_class, test_df_class = grouped_splits(df, test_size=0.2, val_size=0.2, seed=13)

## Metrics

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

clear_output()

## Training

In [ ]:
from transformers import EarlyStoppingCallback

In [ ]:
def train_one_dimension(df_all: pd.DataFrame,  dim_name: str, out_dir: str, label_type = "Discrete_score"):
    # filter to this dimension and clean labels
    df = df_all[df_all["Dimension"].str.strip().str.lower() == dim_name.lower()].copy()
    df[label_type] = df[label_type].str.strip()
    df = df[df[label_type].isin(label_to_num)].copy()
    df["label_id"] = df[label_type].map(label_to_num)

    # splits
    train_df, val_df, test_df = grouped_splits(df, test_size=0.2, val_size=0.2, seed=13)

    # class weights from training distribution (inverse freq)
    counts = train_df["label_id"].value_counts().reindex([0,1,2], fill_value=0).to_numpy()
    freqs = counts / counts.sum()
    inv = 1.0 / np.clip(freqs, 1e-8, None)
    class_weights = torch.tensor(inv / inv.sum() * len(inv), dtype=torch.float32)

    # tokenize
    train_enc = encode_pair(train_df, tok)
    val_enc   = encode_pair(val_df, tok)
    test_enc  = encode_pair(test_df, tok)

    train_ds = Dataset.from_dict(train_enc)
    val_ds   = Dataset.from_dict(val_enc)
    test_ds  = Dataset.from_dict(test_enc)

    # model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, id2label=num_to_label, label2id=label_to_num
    )

    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="best",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        report_to=[],
        save_total_limit=1,
    )

    early_stop = EarlyStoppingCallback(early_stopping_patience=3)

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tok,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[early_stop]
    )

    trainer.train()

    # final eval on test
    test_metrics = trainer.evaluate(test_ds)
    return trainer, test_metrics, counts, (train_df, val_df, test_df)

## Implementation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
import warnings

warnings.filterwarnings('ignore')

### [Final results] Majority-vote coarsening

In [ ]:
results = {}
for dim in dim_names:
    out_dir = f"{dim}_final_new"
    print(dim)
    trainer, test_metrics, train_counts, (train_df, val_df, test_df) = train_one_dimension(
        df, dim, out_dir
    )

    # run prediction to collect per-example outputs
    test_enc = encode_pair(test_df, tok)
    test_ds  = Dataset.from_dict(test_enc)
    pred = trainer.predict(test_ds)

    y_true = pred.label_ids
    y_pred = np.argmax(pred.predictions, axis=-1)

    results[dim] = {
        "metrics": test_metrics,
        "train_counts": train_counts,
        "model_dir": out_dir,
        "y_true": y_true,
        "y_pred": y_pred,
    }

    print('----------------')

Trust


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.084200,0.993696,0.657895,0.501512
2,1.057200,0.970033,0.672932,0.484501
3,0.941800,0.899491,0.684211,0.573675
4,0.622500,1.074551,0.627820,0.531488
5,0.538900,1.005148,0.718045,0.615650
6,0.468600,1.199870,0.736842,0.597694
7,0.235800,1.314120,0.721805,0.612335
8,0.143200,1.449046,0.718045,0.602143


----------------
Sociability


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.120900,0.926891,0.819549,0.522944
2,0.954400,0.808634,0.815789,0.524873
3,0.821600,0.785848,0.778195,0.507881
4,0.618000,0.875751,0.763158,0.501481
5,0.529600,0.990837,0.796992,0.520516


----------------
Competence


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.097400,1.040717,0.578947,0.384205
2,1.046300,0.969943,0.691729,0.509603
3,0.964000,1.010899,0.676692,0.502999
4,0.671400,1.056580,0.661654,0.477110
5,0.522200,1.134320,0.684211,0.505504


----------------


In [ ]:
# Pretty-print
for dim, info in results.items():
    m = info["metrics"]
    print(f"[{dim}] \t | acc={m['eval_accuracy']:.3f}  f1_macro={m['eval_f1_macro']:.3f}  loss={m['eval_loss']:.3f}")

[Trust] 	 | acc=0.789  f1_macro=0.586  loss=1.372
[Sociability] 	 | acc=0.813  f1_macro=0.539  loss=1.234
[Competence] 	 | acc=0.737  f1_macro=0.501  loss=1.133


In [ ]:
# names in the right order
target_names = [num_to_label[i] for i in range(3)]

# reports
for dim, info in results.items():
    print(f"\n[{dim}] classification report")
    print(classification_report(info["y_true"], info["y_pred"],
                                labels=[0,1,2],
                                target_names=target_names,
                                digits=3))

    # confusion matrix
    cm = confusion_matrix(info["y_true"], info["y_pred"], labels=[0,1,2])
    cm_df = pd.DataFrame(cm, index=[f"true_{t}" for t in target_names],
                            columns=[f"pred_{t}" for t in target_names])
    print("\nConfusion matrix:")
    print(cm_df.to_string())


[Trust] classification report
              precision    recall  f1-score   support

         Low      0.849     0.907     0.877       204
     Neutral      0.200     0.136     0.162        22
        High      0.745     0.693     0.718       101

    accuracy                          0.789       327
   macro avg      0.598     0.579     0.586       327
weighted avg      0.773     0.789     0.780       327


Confusion matrix:
              pred_Low  pred_Neutral  pred_High
true_Low           185             4         15
true_Neutral        10             3          9
true_High           23             8         70

[Sociability] classification report
              precision    recall  f1-score   support

         Low      0.906     0.842     0.873       228
     Neutral      0.000     0.000     0.000        15
        High      0.643     0.881     0.744        84

    accuracy                          0.813       327
   macro avg      0.516     0.574     0.539       327
weighted avg  

### [Appendices] Mean-based coarsening

In [ ]:
def coarsen_ww_cat(x):
  if x <= -0.5:
    return 'Low'
  elif x >= 0.5:
    return 'High'
  else:
    return 'Neutral'

In [ ]:
df['Coarse_ww'] = df['Cat_from_mean'].apply(coarsen_ww_cat)

In [ ]:
df['Coarse_ww'].value_counts()

,count
Coarse_ww,
Low,2381
High,1537
Neutral,981


In [ ]:
results_ww = {}
for dim in dim_names:
    out_dir = f"{dim}_three_classes_ww"
    print(dim)
    trainer, test_metrics, train_counts, (train_df, val_df, test_df) = train_one_dimension(
        df, dim, out_dir, label_type = "Coarse_ww"
    )

    # run prediction to collect per-example outputs
    test_enc = encode_pair(test_df, tok)
    test_ds  = Dataset.from_dict(test_enc)
    pred = trainer.predict(test_ds)

    y_true = pred.label_ids
    y_pred = np.argmax(pred.predictions, axis=-1)

    results_ww[dim] = {
        "metrics": test_metrics,
        "train_counts": train_counts,
        "model_dir": out_dir,
        "y_true": y_true,
        "y_pred": y_pred,
    }

    print('----------------')

Trust


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.095600,0.975186,0.639098,0.575033
2,0.917700,0.900092,0.680451,0.575503
3,0.863500,0.849126,0.654135,0.608866
4,0.544300,0.936859,0.684211,0.614189
5,0.463800,0.990047,0.657895,0.599406
6,0.369100,1.148629,0.703008,0.613060
7,0.229100,1.307354,0.684211,0.564383


----------------
Sociability


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.082100,0.907498,0.672932,0.539151
2,0.899500,0.822806,0.687970,0.629596
3,0.842500,0.833292,0.687970,0.639380
4,0.542300,0.870910,0.710526,0.653041
5,0.493100,0.867293,0.672932,0.641692
6,0.357800,0.929986,0.684211,0.637337
7,0.227700,0.999323,0.672932,0.642541


----------------
Competence


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.094900,1.016774,0.545113,0.469209
2,0.963900,0.937684,0.597744,0.574807
3,0.857800,0.982301,0.601504,0.561127
4,0.643200,1.082513,0.586466,0.542372
5,0.535300,1.096918,0.601504,0.582018
6,0.458600,1.129556,0.548872,0.541688
7,0.301000,1.185738,0.571429,0.561735
8,0.267700,1.273705,0.586466,0.568530


----------------


In [ ]:
# Pretty-print
for dim, info in results_ww.items():
    m = info["metrics"]
    print(f"[{dim}] \t | acc={m['eval_accuracy']:.3f}  f1_macro={m['eval_f1_macro']:.3f}  loss={m['eval_loss']:.3f}")

[Trust] 	 | acc=0.749  f1_macro=0.607  loss=1.052
[Sociability] 	 | acc=0.722  f1_macro=0.659  loss=0.950
[Competence] 	 | acc=0.578  f1_macro=0.553  loss=1.207


In [ ]:
# readable names in the right order
target_names = [num_to_label[i] for i in range(3)]

# print reports
for dim, info in results_ww.items():
    print(f"\n[{dim}] classification report")
    print(classification_report(info["y_true"], info["y_pred"],
                                labels=[0,1,2],
                                target_names=target_names,
                                digits=3))

    # optional: confusion matrix for extra diagnostics
    cm = confusion_matrix(info["y_true"], info["y_pred"], labels=[0,1,2])
    cm_df = pd.DataFrame(cm, index=[f"true_{t}" for t in target_names],
                            columns=[f"pred_{t}" for t in target_names])
    print("\nConfusion matrix:")
    print(cm_df.to_string())


[Trust] classification report
              precision    recall  f1-score   support

         Low      0.867     0.844     0.856       186
     Neutral      0.364     0.186     0.246        43
        High      0.645     0.816     0.721        98

    accuracy                          0.749       327
   macro avg      0.625     0.615     0.607       327
weighted avg      0.735     0.749     0.735       327


Confusion matrix:
              pred_Low  pred_Neutral  pred_High
true_Low           157             7         22
true_Neutral        13             8         22
true_High           11             7         80

[Sociability] classification report
              precision    recall  f1-score   support

         Low      0.860     0.790     0.824       186
     Neutral      0.301     0.510     0.379        49
        High      0.877     0.696     0.776        92

    accuracy                          0.722       327
   macro avg      0.679     0.665     0.659       327
weighted avg  

In [ ]:
results_ww['Trust']['metrics']

{'eval_loss': 1.05189049243927,
 'eval_accuracy': 0.7492354740061162,
 'eval_f1_macro': 0.6074867993124123,
 'eval_runtime': 2.1669,
 'eval_samples_per_second': 150.91,
 'eval_steps_per_second': 5.076,
 'epoch': 7.0}

# **BERT Regression Models**

## Imports

In [ ]:
!pip install evaluate
from dataclasses import dataclass
from typing import Dict, List
import numpy as np
from scipy.stats import spearmanr
import pandas as pd
import torch
import evaluate

from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from transformers import EarlyStoppingCallback
from datasets import Dataset

!pip install emoji==0.6.0
import emoji

model_name = "vinai/bertweet-base"
tok = AutoTokenizer.from_pretrained(model_name, use_fast=False, normalization=True)

clear_output()

In [ ]:
dim_names = ["Trust", "Sociability", "Competence"]

## Tokenization

In [ ]:
def encode_pair(df: pd.DataFrame, tokenizer, max_length=256):
    enc = tokenizer(
        df["Text"].tolist(),
        text_pair=df["Target"].tolist(),
        truncation=True, padding="max_length", max_length=max_length
    )
    enc["labels"] = df["score"].astype(np.float32).tolist()
    return enc

In [ ]:
def grouped_splits(df: pd.DataFrame, test_size=0.2, val_size=0.2, seed=13):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss.split(df, groups=df["Text"]))
    trainval = df.iloc[trainval_idx].reset_index(drop=True)
    test     = df.iloc[test_idx].reset_index(drop=True)

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx, val_idx = next(gss2.split(trainval, groups=trainval["Text"]))
    train = trainval.iloc[train_idx].reset_index(drop=True)
    val   = trainval.iloc[val_idx].reset_index(drop=True)
    return train, val, test

## Training and metrics

In [ ]:
def round_to_seven_classes(values):
    x = np.asarray(values, dtype=float)
    return np.select(
        [
            x >= 2.5,
            x >= 1.5,
            x >= 0.5,
            x >= -0.5,
            x >= -1.5,
            x >= -2.5
        ],
        [3, 2, 1, 0, -1, -2],
        default=-3
    ).astype(int)

In [ ]:
class RegTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").view(-1, 1)
        outputs = model(**inputs)
        preds = outputs.logits
        loss_fn = torch.nn.SmoothL1Loss(beta=1.0)
        loss = loss_fn(preds, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics_reg(eval_pred):
    preds, labels = eval_pred
    y_pred = preds.reshape(-1)
    y_true = labels.reshape(-1)
    mae  = float(np.mean(np.abs(y_pred - y_true)))
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    sp   = float(spearmanr(y_true, y_pred, nan_policy="omit").statistic)
    return {"mae": mae, "rmse": rmse, "spearman": sp}

In [ ]:
def train_regressor_one_dim(df_all: pd.DataFrame, dim_name: str, out_dir: str):
    # filter this dimension and prepare the continuous label
    df = df_all[df_all["Dimension"].str.strip() == dim_name].copy()
    df = df.dropna(subset=[score_col])
    df["score"] = df[score_col].astype(float).clip(-3, 3)

    # splits
    train_df, val_df, test_df = grouped_splits(df, test_size=0.2, val_size=0.2, seed=13)

    # tokenize
    train_enc = encode_pair(train_df, tok)
    val_enc   = encode_pair(val_df, tok)
    test_enc  = encode_pair(test_df, tok)

    train_ds = Dataset.from_dict(train_enc)
    val_ds   = Dataset.from_dict(val_enc)
    test_ds  = Dataset.from_dict(test_enc)

    # model: regression head
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=1
    )
    model.config.problem_type = "regression"

    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=1e-5,
        weight_decay=0.01,
        warmup_ratio=0.06,
        eval_strategy="epoch",
        save_strategy="best",
        load_best_model_at_end=True,
        metric_for_best_model="mae",
        greater_is_better=False,
        save_total_limit=1,
        logging_steps=50,
        report_to=[]
    )

    early_stop = EarlyStoppingCallback(early_stopping_patience=3)

    trainer = RegTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tok,
        compute_metrics=compute_metrics_reg,
        callbacks=[early_stop]
    )

    trainer.train()

    # test metrics + per-example predictions for later reports
    test_metrics = trainer.evaluate(test_ds)
    pred = trainer.predict(test_ds)
    y_true_cont = pred.label_ids.reshape(-1)
    y_pred_cont = pred.predictions.reshape(-1)

    # derive discrete class ids for a 7-class report
    y_true_disc = round_to_seven_classes(y_true_cont)
    y_pred_disc = round_to_seven_classes(y_pred_cont)

    return {
        "trainer": trainer,
        "metrics": test_metrics,
        "y_true_cont": y_true_cont,
        "y_pred_cont": y_pred_cont,
        "y_true_disc": y_true_disc,
        "y_pred_disc": y_pred_disc,
        "test_df": test_df
    }

## Final Results

In [ ]:
regression_res['Trust_mean_true_ww'] = regression_res['Trust_mean_true'].copy().apply(coarsen_ww)
regression_res['Sociability_mean_true_ww'] = regression_res['Sociability_mean_true '].copy().apply(coarsen_ww)
regression_res['Competence_mean_true_ww'] = regression_res['Competence_mean_true'].copy().apply(coarsen_ww)

return_stats(regression_res['Trust_mean_true_ww'], regression_res['Trust_mean_pred'])
return_stats(regression_res['Sociability_mean_true_ww'], regression_res['Sociability_mean_pred'])
return_stats(regression_res['Competence_mean_true_ww'], regression_res['Competence_mean_pred'])

0.35 	 0.31 	 0.83 	 0.43 	 0.32
0.46 	 0.34 	 0.88 	 0.33 	 0.35
0.36 	 0.24 	 0.86 	 0.26 	 0.26


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# **Zero-Shot Prompts**

## System messages

### Fine-grained

In [ ]:
import tqdm
from tqdm import tqdm

In [ ]:
system_message = """
You are a professional language analyst and sociologist.
You work on warmth and competence, which are two concepts in sociology.
You know that warmth is decomposed into trust, which relates to the moral and personal aspect of the target; and sociability, which relates to the relational and societal aspect and impact of the target.
You work on a sentence-level: you read the full sentence, with all its components, before you assess it
There is a degree of subjectivity in this task, so you consider the meaning that the general population would agree with and consider.
You do not add information that does not appear in the text.
The context matters the most to you. You consider all relevant information.
You understand that sarcasm is an ironic remark meant to mock by saying something different than what the speaker really means.
You understand that irony is the humorous or mildly sarcastic use of words to imply the opposite of what they normally mean.
You understand that hyperbole is the extreme, dramatic exaggeration for effect, not meant to be taken literally.
You understand how irony, sarcasm, and hyperbole are employed in social media and in slang, and that the literal wording does not always reflect the true meaning and sentiment.
You assess warmth and competence on a scale from -3 to +3.
You adhere to the provided target. Meaning, you understand that even if the speaker is explicitly expressing opinions towards entity_X, you give the score for trust towards entity_Y only if the target given to you is entity_Y.
You do not use first-person pronouns such as "I" or "we" in your answer.
You adhere to the targets given to you in your assessment of warmth or competence.
You understand that the use of hashtags in social media can be complementary to the text. However, that is not always the case, as hashtags can be used to categorize content, increase visibility, and connect users with relevant discussions and communities, rather than as sentiment markers.
You understand that when the target is "religious people", then it refers to religions and those who believe in any God or practice any religion.
You understand that when the target is "nonreligious people", then it refers to atheism and those who are not religious (i.e., atheists/agnostics).
You understand that when the target is "women", then it refers to women or girls and/or interconnected topics: sexism, feminism, misogyny, bias, and female representation.
You understand that when the target is "Hillary Clinton", then it refers to the 2016 US presidential candidate and former Secretary of State Hillary Clinton.
You understand that when the target is "Donald Trump", then it refers to the 2016 US presidential candidate and current US president Donald Trump.
You understand that when the target is "Barack Obama", then it refers to the former US president Barack Obama.
You understand that when the target is "Environmentalists", then it refers to to environment and climate change activists.
"""

### Coarse-grained

In [ ]:
system_message_coarse = """

You are a professional language analyst and sociologist.

You work on warmth and competence, which are two concepts in sociology.

You know that warmth is decomposed into trust, which relates to the moral and personal aspect of the target; and sociability, which relates to the relational and societal aspect and impact of the target.

You work on a sentence-level: you read the full sentence, with all its components, before you assess it
There is a degree of subjectivity in this task, so you consider the meaning that the general population would agree with and consider.

You do not add information that does not appear in the text.

The context matters the most to you. You consider all relevant information.

You understand that sarcasm is an ironic remark meant to mock by saying something different than what the speaker really means.

You understand that irony is the humorous or mildly sarcastic use of words to imply the opposite of what they normally mean.

You understand that hyperbole is the extreme, dramatic exaggeration for effect, not meant to be taken literally.

You understand how irony, sarcasm, and hyperbole are employed in social media and in slang, and that the literal wording does not always reflect the true meaning and sentiment.

You assess warmth and competence on the scale of "low", "neutral, not applicable, not expressed", and "high".

You adhere to the provided target. Meaning, you understand that even if the speaker is explicitly expressing opinions towards entity_X, you give the score for trust towards entity_Y only if the target given to you is entity_Y.

You do not use first-person pronouns such as "I" or "we" in your answer.

You adhere to the targets given to you in your assessment of warmth or competence.

You understand that the use of hashtags in social media can be complementary to the text. However, that is not always the case, as hashtags can be used to categorize content, increase visibility, and connect users with relevant discussions and communities, rather than as sentiment markers.

You understand that when the target is "religious people", then it refers to religions and those who believe in any God or practice any religion.

You understand that when the target is "nonreligious people", then it refers to atheism and those who are not religious (i.e., atheists/agnostics).

You understand that when the target is "women", then it refers to women or girls and/or interconnected topics: sexism, feminism, misogyny, bias, and female representation.

You understand that when the target is "Hillary Clinton", then it refers to the 2016 US presidential candidate and former Secretary of State Hillary Clinton.

You understand that when the target is "Donald Trump", then it refers to the 2016 US presidential candidate and current US president Donald Trump.

You understand that when the target is "Barack Obama", then it refers to the former US president Barack Obama.

You understand that when the target is "Environmentalists", then it refers to to environment and climate change activists.

"""

## Trust

### Fine-grained

In [ ]:
user_role_trust = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).
Recently, psychologists have modelled warmth through two dimensions: trust (T) and sociability (S).
Your task is to assess the degree of trust towards a specific target within a sentence.

The focus in this dimension is on the personal / moral aspect of the target.
High trust can be defined as morality, kindness, sincerity, trustworthiness, and honesty.
Words associated with high trust: charity, mother, compliment, affectionate.
Low trust can be defined as immorality, insincerity, dishonesty, untrustworthiness, dubiousness, and maliciousness.
Words associated with low trust: discredit, bribe, espionage, disinformation, disloyal.

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).
Rate the apparent level of trust that the sentence's author seems to express towards the specified target.
Choose one label from these seven labels:
high distrust\n
moderate distrust\n
slight distrust\n
neutral, not applicable, not expressed\n
slight trust\n
moderate trust\n
high trust\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess trust, try to answer the following questions:\n
1. What is the degree of trust towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} as trustworthy or untrustworthy / moral or immoral / honest or dishonest?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of the trust towards {} only.
In the format of a JSON file or a Python dictionary, you should provide your justification saved in a key called "reason". Then, based on your justification, add your rating to a key called "label".

"""

### Coarse-grained

In [ ]:
user_role_trust_coarse = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).
Recently, psychologists have modelled warmth through two dimensions: trust (T) and sociability (S).
Your task is to assess the degree of trust towards a specific target within a sentence.

The focus in this dimension is on the personal and moral aspect of the target.
High trust can be defined as morality, kindness, sincerity, trustworthiness, and honesty.
Words associated with high trust: charity, mother, compliment, affectionate.
Low trust can be defined as immorality, insincerity, dishonesty, untrustworthiness, dubiousness, and maliciousness.
Words associated with low trust: discredit, bribe, espionage, disinformation, disloyal.

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).
Rate the apparent level of trust that the sentence's author seems to express towards the specified target.

Choose one label from these three labels:

negative trust (which can include slight, moderate, or high distrust).\n
neutral, not expressed, not applicable\n
positive trust (which can include slight, moderate, or high trust).\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess trust, try to answer the following questions:\n
1. What is the degree of trust towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} as trustworthy or untrustworthy / moral or immoral / honest or dishonest?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of the trust towards {} only.
You should analyse the meaning, then, based on your analysis, add your rating to a key called "label".
You should provide your label in a JSON object whose key is called "label" and that label only.

"""

## Sociability

### Fine-grained

In [ ]:
user_role_soc = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).\n
Recently,   have modelled warmth through two dimensions: trust (T) and sociability (S).\n
Your task is to assess the degree of trust towards a specific target within a sentence.\n
The focus in this dimension is on the social aspect of the target and its relational impact on others or society as a whole.\n
High sociability can be defined as friendliness, sociableness, generosity, and helpfulness.\n
Words associated with high sociability: helpful, intimate, laugh, celebration, reliant, entertain, social club, bestie.\n
Low sociability can be defined as antisocial behavior, lack of generosity, inconsiderateness, indifference, and unhelpfulness.\n
Words associated with low sociability: ingrate, abduct, selfish, theft, egomaniacal, pervert.\n
Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).\n
Rate the apparent level of sociability that the sentence's author seems to express towards the specified target.\n
Choose one label from these seven labels:
high unsociability\n
moderate unsociability\n
slight unsociability\n
neutral, not applicable, not expressed, etc.\n
slight sociability\n
moderate sociability\n
high sociability\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess sociability, try to answer the following questions:\n
1. What is the degree of sociability towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} as sociable or antisocial? Helpful or unhelpful?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of perceived sociability trust towards {} only.
In the format of a JSON file or a Python dictionary, you should provide your justification saved in a key called "reason". Then, based on your justification, add your rating to a key called "label".
"""

### Coarse-grained

In [ ]:
user_role_soc_coarse = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).\n
Recently,   have modelled warmth through two dimensions: trust (T) and sociability (S).\n
Your task is to assess the degree of trust towards a specific target within a sentence.\n
The focus in this dimension is on the social aspect of the target and its relational impact on others or society as a whole.\n
High sociability can be defined as friendliness, sociableness, generosity, and helpfulness.\n
Words associated with high sociability: helpful, intimate, laugh, celebration, reliant, entertain, social club, bestie.\n
Low sociability can be defined as antisocial behavior, lack of generosity, inconsiderateness, indifference, and unhelpfulness.\n
Words associated with low sociability: ingrate, abduct, selfish, theft, egomaniacal, pervert.\n

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).\n
Rate the apparent level of sociability that the sentence's author seems to express towards the specified target.\n
Choose one label from these three labels:

negative sociability (which can include slight, moderate, or high unsociability).
neutral, not expressed, not applicable\n
positive sociability (which can include slight, moderate, or high sociability).


Here is the sentence: {}.\n
Its target is: {}.

In order to assess sociability, try to answer the following questions:\n
1. What is the degree of sociability towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} as sociable or antisocial? Helpful or unhelpful?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of perceived sociability trust towards {} only.
You should analyse the meaning, then, based on your analysis, add your rating to a key called "label".
You should provide your label in a JSON object whose key is called "label" and that label only.
"""

## Competence

### Fine-grained

In [ ]:
user_role_comp = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).\n
Your task is to assess the degree of competence towards a specific target within a sentence.\n
Competence, in the broader sense, is interchangeable with 'ability' or 'capability'.
Competence can be defined as ability, power, dominance, being in control, importance, having influence, and assertiveness.\n
Words associated with high competence: hitman, heroical, entrepreneurship, strategies, superman, viper, impunity.\n
Incompetence can be defined as submissiveness, not being in control of a situation, being controlled or guided by outside factors, or weakness.\n
Words associated with low competence: bootlicker, talentless, crash landing, bedridden, underfed.\n
Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).\n
Rate the apparent level of competence that the sentence's author seems to express towards the specified target.\n
Choose one label from these seven labels:
high incompetence\n
moderate incompetence\n
slight incompetence\n
neutral, not applicable, not expressed\n
slight competence\n
moderate competence\n
high competence\n
Adhere to the literal meaning of competence, which may be "positive" (e.g., a CEO) or "negative" (e.g., a villain or a dictator). Both types are considered competence, regardless of the outcomes.\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess competence, try to answer the following questions:\n
1. What is the degree of competence towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} in control or out of control? Active or passive? Powerful or weak?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of the competence towards {} only.
In the format of a JSON file or a Python dictionary, you should provide your justification saved in a key called "reason". Then, based on your justification, add your rating to a key called "label".
"""

### Coarse-grained

In [ ]:
user_role_comp_coarse = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).\n
Your task is to assess the degree of competence towards a specific target within a sentence.\n
Competence, in the broader sense, is interchangeable with 'ability' or 'capability'.
Competence can be defined as ability, power, dominance, being in control, importance, having influence, and assertiveness.\n
Words associated with high competence: hitman, heroical, entrepreneurship, strategies, superman, viper, impunity.\n
Incompetence can be defined as submissiveness, not being in control of a situation, being controlled or guided by outside factors, or weakness.\n
Words associated with low competence: bootlicker, talentless, crash landing, bedridden, underfed.\n

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).\n

Rate the apparent level of competence that the sentence's author seems to express towards the specified target.\n
Choose one of these three labels:
negative competence (which can include slight, moderate, or high incompetence).
neutral, not expressed, not applicable\n
positive competence (which can include slight, moderate, or high competence).

Adhere to the literal meaning of competence, which may be "positive" (e.g., a CEO) or "negative" (e.g., a villain or a dictator). Both types are considered competence, regardless of the outcomes.\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess competence, try to answer the following questions:\n
1. What is the degree of competence towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} in control or out of control? Active or passive? Powerful or weak?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of the competence towards {} only.
You should analyse the meaning, then, based on your analysis, add your rating to a key called "label".
You should provide your label in a JSON object whose key is called "label" and that label only.
"""

# **Few-Shot Prompts**

## Trust

### Fine-grained

In [ ]:
user_role_trust_fs = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).
Recently, psychologists have modelled warmth through two dimensions: trust (T) and sociability (S).
Your task is to assess the degree of trust towards a specific target within a sentence.

The focus in this dimension is on the personal / moral aspect of the target.
High trust can be defined as morality, kindness, sincerity, trustworthiness, and honesty.
Words associated with high trust: charity, mother, compliment, affectionate.
Low trust can be defined as immorality, insincerity, dishonesty, untrustworthiness, dubiousness, and maliciousness.
Words associated with low trust: discredit, bribe, espionage, disinformation, disloyal.

Here are five examples, each containing the text, the target, the score, and the rationale behind the score.\n

Example 1:\n

Target: Women\n
Text: My wife is the most caring person I've ever met ... she's the only woman in a house full of testosterone . She never stops working whether it's at home or being an RN . I cant keep up but I try . She makes me a better person . I'd be lost without her . Oh and she's smoking hot too.\n
Trust score: +3 (high trust). \n
Rationale: The author expresses maximum trust in women through his wife as a representative example. He portrays his wife as a trustworthy, dependable, and caring individual who is essential to his well-being ("I'd be lost without her"). The statement "she makes me a better person" implies the author views his wife as having a strong moral character that positively influences him.\n

Example 2:\n

Target: US Presidential Candidate Hillary Clinton \n
Text: Would you wanna be in a long term relationship with some bitch that hides her emails, & lies to your face? Then Dont vote\n
Trust score: -3 (high distrust). \n
Rationale: The author explicitly portrays Clinton as someone who is fundamentally untrustworthy and cannot be relied upon to tell the truth or be transparent through two direct accusations: "hides her emails" and "lies to your face." With concealment and deception some of the strongest markers of untrustworthiness, the author emphasizes these as core trust violations. \n

Example 3:\n

Target: Women \n
Text: I need feminism because the United States is one of the only countries that doesn't give paid maternity leave.\n
Trust score: 0 (neutral). \n
Rationale: The author’s statement is focused on policy rather than character traits and doesn't make any attributions about women's trustworthiness or morality. This is a policy statement rather than a personal attribution about women's trustworthiness. This should help you distinguish between advocacy/policy statements and personal/characteristic attributions.\n

Example 4:\n

Target: Religious people \n
Text: Could all those who believe in a god please leave. The meeting will now continue for the grown ups only.\n
Trust score: 0 (neutral).\n
Rationale: The criticism is entirely focused on intellectual maturity rather than character or morality. The author is suggesting that religious people shouldn't participate in this particular discussion. On the other hand, the author doesn't make any claims about religious people's morality, honesty, sincerity, or trustworthiness. There are no accusations of deception, dishonesty, or moral failings. \n

Example 5:\n

Target: Women\n
Text: "when i was 16 i had a folder of “ feel good songs ” and everyday i would select one and send it to my best friend along with a paragraph of how much they meant to me and why i they should be happy and then i would lay in bed thinking “ no i’m not gay this is just what girls do :) ”"\n
Trust score: +3 (high trust).\n
Rationale: The author attributes sincere, genuine intentions to women through the described behaviors. The daily practice of sending heartfelt messages "about how much they meant to me and why they should be happy" suggests honesty and sincerity in women’s relationships. The author portrays women as having good moral intentions in their friendships by genuinely caring about others' wellbeing. \n

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).
Rate the apparent level of trust that the sentence's author seems to express towards the specified target.
Choose one label from these seven labels:
high distrust\n
moderate distrust\n
slight distrust\n
neutral, not applicable, not expressed\n
slight trust\n
moderate trust\n
high trust\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess trust, try to answer the following questions:\n
1. What is the degree of trust towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} as trustworthy or untrustworthy / moral or immoral / honest or dishonest?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of the trust towards {} only.
You should provide your label in a JSON format whose key is called "label" and that label only. Do not provide your rationale.

"""

### Coarse-grained

In [ ]:
user_role_trust_fs_coarse = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).
Recently, psychologists have modelled warmth through two dimensions: trust (T) and sociability (S).
Your task is to assess the degree of trust towards a specific target within a sentence.

The focus in this dimension is on the personal / moral aspect of the target.
High trust can be defined as morality, kindness, sincerity, trustworthiness, and honesty.
Words associated with high trust: charity, mother, compliment, affectionate.
Low trust can be defined as immorality, insincerity, dishonesty, untrustworthiness, dubiousness, and maliciousness.
Words associated with low trust: discredit, bribe, espionage, disinformation, disloyal.

Here are five examples, each containing the text, the target, the score, and the rationale behind the score.\n

Example 1:\n

Target: Women\n
Text: My wife is the most caring person I've ever met ... she's the only woman in a house full of testosterone . She never stops working whether it's at home or being an RN . I cant keep up but I try . She makes me a better person . I'd be lost without her . Oh and she's smoking hot too.\n
Trust score: positive trust. \n
Rationale: The author expresses maximum trust in women through his wife as a representative example. He portrays his wife as a trustworthy, dependable, and caring individual who is essential to his well-being ("I'd be lost without her"). The statement "she makes me a better person" implies the author views his wife as having a strong moral character that positively influences him.\n

Example 2:\n

Target: US Presidential Candidate Hillary Clinton \n
Text: Would you wanna be in a long term relationship with some bitch that hides her emails, & lies to your face? Then Dont vote\n
Trust score: negative trust. \n
Rationale: The author explicitly portrays Clinton as someone who is fundamentally untrustworthy and cannot be relied upon to tell the truth or be transparent through two direct accusations: "hides her emails" and "lies to your face." With concealment and deception some of the strongest markers of untrustworthiness, the author emphasizes these as core trust violations. \n

Example 3:\n

Target: Women \n
Text: I need feminism because the United States is one of the only countries that doesn't give paid maternity leave.\n
Trust score: neutral. \n
Rationale: The author’s statement is focused on policy rather than character traits and doesn't make any attributions about women's trustworthiness or morality. This is a policy statement rather than a personal attribution about women's trustworthiness. This should help you distinguish between advocacy/policy statements and personal/characteristic attributions.\n

Example 4:\n

Target: Religious people \n
Text: Could all those who believe in a god please leave. The meeting will now continue for the grown ups only.\n
Trust score: neutral.\n
Rationale: The criticism is entirely focused on intellectual maturity rather than character or morality. The author is suggesting that religious people shouldn't participate in this particular discussion. On the other hand, the author doesn't make any claims about religious people's morality, honesty, sincerity, or trustworthiness. There are no accusations of deception, dishonesty, or moral failings. \n

Example 5:\n

Target: Women\n
Text: "when i was 16 i had a folder of “ feel good songs ” and everyday i would select one and send it to my best friend along with a paragraph of how much they meant to me and why i they should be happy and then i would lay in bed thinking “ no i’m not gay this is just what girls do :) ”"\n
Trust score: positive trust.\n
Rationale: The author attributes sincere, genuine intentions to women through the described behaviors. The daily practice of sending heartfelt messages "about how much they meant to me and why they should be happy" suggests honesty and sincerity in women’s relationships. The author portrays women as having good moral intentions in their friendships by genuinely caring about others' wellbeing. \n

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).
Rate the apparent level of trust that the sentence's author seems to express towards the specified target.
Choose one label from these three labels:

negative trust (which can include slight, moderate, or high distrust).
neutral, not expressed, not applicable\n
positive trust (which can include slight, moderate, or high trust).

Here is the sentence: {}.\n
Its target is: {}.

In order to assess trust, try to answer the following questions:\n
1. What is the degree of trust towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} as trustworthy or untrustworthy / moral or immoral / honest or dishonest?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of the trust towards {} only.
You should analyse the meaning, then, based on your analysis, add your rating to a key called "label".
You should provide your label in a JSON object whose key is called "label" and that label only.
"""

## Sociability

### Fine-grained

In [ ]:
user_role_soc_fs = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).\n
Recently, psychologists have modelled warmth through two dimensions: trust and sociability.\n
Your task is to assess the degree of sociability towards a specific target within a sentence.\n
The focus in this dimension is on the social aspect of the target and its relational impact on others or society as a whole.\n
High sociability can be defined as friendliness, sociableness, generosity, and helpfulness.\n
Words associated with high sociability: helpful, intimate, laugh, celebration, reliant, entertain, social club, bestie.\n
Low sociability can be defined as antisocial behavior, lack of generosity, inconsiderateness, indifference, and unhelpfulness.\n
Words associated with low sociability: ingrate, abduct, selfish, theft, egomaniacal, pervert.\n

Here are five examples, each containing the text, the target, the score, and the rationale behind the score.\n

Example 1:\n

Target: Women\n
Text: My wife is the most caring person I've ever met ... she's the only woman in a house full of testosterone . She never stops working whether it's at home or being an RN . I cant keep up but I try . She makes me a better person . I'd be lost without her . Oh and she's smoking hot too.\n

Sociability score: +3 (high sociability). \n

Rationale: The text attributes maximum sociability to women through his wife. "Most caring person I've ever met" shows high sociability and interpersonal warmth. He emphasizes how essential her social/emotional qualities are to their family dynamic, particularly in a "house full of testosterone." The statement "whether it's at home or being an RN" suggests helpfulness and generosity both personally and professionally. \n

Example 2:\n

Target: US Presidential Candidate Hillary Clinton \n
Text: Would you wanna be in a long term relationship with some bitch that hides her emails, & lies to your face? Then Dont vote\n

Sociability score: -2 (moderate unsociability). \n

Rationale: The derogatory term "bitch" (which is demeaning towards women) and the comparison to an undesirable romantic partner frames Clinton as someone who would be unpleasant to be around or interact with socially. The rhetorical question implies she would be toxic in close social relationships. However, one can view the sociability attack is more about being unpleasant in relationships rather than being completely antisocial or unhelpful in all social contexts, hence a maximum score was not assigned.

Example 3:\n

Target: Women\n
Text: I need feminism because the United States is one of the only countries that doesn't give paid maternity leave.\n
Sociability score: 0 (neutral). \n
Rationale: The author’s statement is focused on policy rather than character traits. There is no commentary on women's interpersonal qualities or social behavior. Note: This is a policy statement rather than a personal attribution about women's sociability. This should help you distinguish between advocacy/policy statements and personal/characteristic attributions. \n

Example 4:\n

Target: Religious people\n
Text: Could all those who believe in a god please leave. The meeting will now continue for the grown ups only.\n
Sociability score: -3 (high unsociability). \n
Rationale: The author portrays religious people as socially incompetent and needing exclusion from adult discourse ("please leave") and says their presence is incompatible with or unwanted in serious adult conversations ("grown ups only"). The order (telling an entire social group to leave) and the dismissive language are extremely exclusionary and socially hostile, resulting in a maximum unsociability score.\n

Example 5:\n

Target: Women
Text: when i was 16 i had a folder of “feel good songs” and everyday i would select one and send it to my best friend along with a paragraph of how much they meant to me and why i they should be happy and then i would lay in bed thinking “ no i’m not gay this is just what girls do :) ”\n
Sociability score: +3 (high sociability). \n
Rationale: The text attributes maximum sociability to women. The actions described (sending feel-good songs, writing paragraphs about how much someone means to you, wanting to make someone happy) represent peak social engagement and helpfulness. The phrase "this is just what girls do" frames these highly sociable, caring behaviors as naturally feminine traits and hence portrays women as exceptionally generous with their emotional/social energy.\n

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).\n
Rate the apparent level of sociability that the sentence's author seems to express towards the specified target.\n
Choose one label from these seven labels:
high unsociability\n
moderate unsociability\n
slight unsociability\n
neutral, not applicable, not expressed, etc.\n
slight sociability\n
moderate sociability\n
high sociability\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess sociability, try to answer the following questions:\n
1. What is the degree of sociability towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} as sociable or antisocial? Helpful or unhelpful?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of perceived sociability trust towards {} only.
You should provide your label in a JSON format whose key is called "label" and that label only. Do not provide your rationale.
"""

### Coarse-grained

In [ ]:
user_role_soc_fs_coarse = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).\n
Recently, psychologists have modelled warmth through two dimensions: trust and sociability.\n
Your task is to assess the degree of sociability towards a specific target within a sentence.\n
The focus in this dimension is on the social aspect of the target and its relational impact on others or society as a whole.\n
High sociability can be defined as friendliness, sociableness, generosity, and helpfulness.\n
Words associated with high sociability: helpful, intimate, laugh, celebration, reliant, entertain, social club, bestie.\n
Low sociability can be defined as antisocial behavior, lack of generosity, inconsiderateness, indifference, and unhelpfulness.\n
Words associated with low sociability: ingrate, abduct, selfish, theft, egomaniacal, pervert.\n

Here are five examples, each containing the text, the target, the score, and the rationale behind the score.\n

Example 1:\n

Target: Women\n
Text: My wife is the most caring person I've ever met ... she's the only woman in a house full of testosterone . She never stops working whether it's at home or being an RN . I cant keep up but I try . She makes me a better person . I'd be lost without her . Oh and she's smoking hot too.\n

Sociability score: positive sociability. \n

Rationale: The text attributes maximum sociability to women through his wife. "Most caring person I've ever met" shows high sociability and interpersonal warmth. He emphasizes how essential her social/emotional qualities are to their family dynamic, particularly in a "house full of testosterone." The statement "whether it's at home or being an RN" suggests helpfulness and generosity both personally and professionally. \n

Example 2:\n

Target: US Presidential Candidate Hillary Clinton \n
Text: Would you wanna be in a long term relationship with some bitch that hides her emails, & lies to your face? Then Dont vote\n

Sociability score: negative sociability. \n

Rationale: The derogatory term "bitch" (which is demeaning towards women) and the comparison to an undesirable romantic partner frames Clinton as someone who would be unpleasant to be around or interact with socially. The rhetorical question implies she would be toxic in close social relationships. However, one can view the sociability attack is more about being unpleasant in relationships rather than being completely antisocial or unhelpful in all social contexts, hence a maximum score was not assigned.

Example 3:\n

Target: Women\n
Text: I need feminism because the United States is one of the only countries that doesn't give paid maternity leave.\n
Sociability score: neutral. \n
Rationale: The author’s statement is focused on policy rather than character traits. There is no commentary on women's interpersonal qualities or social behavior. Note: This is a policy statement rather than a personal attribution about women's sociability. This should help you distinguish between advocacy/policy statements and personal/characteristic attributions. \n

Example 4:\n

Target: Religious people\n
Text: Could all those who believe in a god please leave. The meeting will now continue for the grown ups only.\n
Sociability score: negative sociability. \n
Rationale: The author portrays religious people as socially incompetent and needing exclusion from adult discourse ("please leave") and says their presence is incompatible with or unwanted in serious adult conversations ("grown ups only"). The order (telling an entire social group to leave) and the dismissive language are extremely exclusionary and socially hostile, resulting in a maximum unsociability score.\n

Example 5:\n

Target: Women
Text: when i was 16 i had a folder of “feel good songs” and everyday i would select one and send it to my best friend along with a paragraph of how much they meant to me and why i they should be happy and then i would lay in bed thinking “ no i’m not gay this is just what girls do :) ”\n
Sociability score: positive sociability. \n
Rationale: The text attributes maximum sociability to women. The actions described (sending feel-good songs, writing paragraphs about how much someone means to you, wanting to make someone happy) represent peak social engagement and helpfulness. The phrase "this is just what girls do" frames these highly sociable, caring behaviors as naturally feminine traits and hence portrays women as exceptionally generous with their emotional/social energy.\n

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).\n
Rate the apparent level of sociability that the sentence's author seems to express towards the specified target.\n

Choose one label from these three labels:

negative sociability (which can include slight, moderate, or high unsociability).
neutral, not expressed, not applicable\n
positive sociability (which can include slight, moderate, or high sociability).

Here is the sentence: {}.\n
Its target is: {}.

In order to assess sociability, try to answer the following questions:\n
1. What is the degree of sociability towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} as sociable or antisocial? Helpful or unhelpful?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of perceived sociability trust towards {} only.
You should analyse the meaning, then, based on your analysis, add your rating to a key called "label".
You should provide your label in a JSON object whose key is called "label" and that label only.
"""

## Competence

### Fine-grained

In [ ]:
user_role_comp_fs = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).\n
Your task is to assess the degree of competence towards a specific target within a sentence.\n
Competence, in the broader sense, is interchangeable with 'ability' or 'capability'.
Competence can be defined as ability, power, dominance, being in control, importance, having influence, and assertiveness.\n
Words associated with high competence: hitman, heroical, entrepreneurship, strategies, superman, viper, impunity.\n
Incompetence can be defined as submissiveness, not being in control of a situation, being controlled or guided by outside factors, or weakness.\n
Words associated with low competence: bootlicker, talentless, crash landing, bedridden, underfed.\n

Here are five examples, each containing the text, the target, the score, and the rationale behind the score.\n

Example 1:\n
Target: Women\n
Text: “My wife is the most caring person I've ever met ... she's the only woman in a house full of testosterone . She never stops working whether it's at home or being an RN . I cant keep up but I try . She makes me a better person . I'd be lost without her . Oh and she's smoking hot too.”\n
Competence score: +3 (high competence).\n
Rationale: The author says that his wife “never stops working” in the context of her role as an RN (registered nurse) and as a mother, which suggests multitasking, capability, and professional competence. “I can't keep up but I try” indicates a highly active and energetic individual.\n

Example 2:\n
Target: Hillary Clinton \n
Text: “Would you wanna be in a long term relationship with some bitch that hides her emails, & lies to your face? Then Dont vote \n
Competence score: +2 (moderate competence).\n
Rationale: Despite the author implying that Clinton is manipulative and dishonest, the author’s phrasing doesn't suggest that she's powerless or ineffective, as the negative behaviors described (concealment and deception) require some degree of agency and planning. The author suggests that Clinton is being deliberate in her (“negative”) performance and active in her (“negative”) effects, leading to a moderately high score in the competence dimension.

Example 3:\n
Target: Women\n
Text: I need feminism because the United States is one of the only countries that doesn't give paid maternity leave.\n
Competence score: 0 (neutral).\n
Rationale: The statement implies that women deserve certain rights/benefits, but it doesn't directly attribute dominance or control to women. The statement doesn't characterize women as active, powerful, passive, or weak. The author is advocating for institutional change (paid maternity leave) rather than making claims about women's power or capabilities. This is a policy statement rather than a personal attribution about women's competence. This should help you distinguish between advocacy/policy statements and personal/characteristic attributions.

Example 4:\n
Target: Religious people\n
Text: Could all those who believe in a god please leave. The meeting will now continue for the grown ups only.\n
Competence score: -3 (high incompetence). \n
Rationale: Religious people are portrayed as needing to be excluded from decision-making processes, suggesting they lack the authority or standing or cognitive abilities to participate in important discussions. Additionally, the "grown ups only" framing explicitly characterizes religious people as childlike, passive, and intellectually weak (i.e., subordinates to "grown ups”). This is a direct attack on their mental capacity and maturity, which are core competence attributes.\n

Example 5:\n
Target: Women\n
Text: What the fuck do women even do? I mean seriously they're just useless other than sex. #womensrights #Feminist \n
Competence score: -3 (high incompetence). \n
Rationale: The text explicitly depicts women as “useless” and valuable only for sex, which strips them of any functional, intellectual, or productive role. This framing directly denies women competence by asserting they contribute nothing meaningful beyond sexual utility. By reducing women to a single, degrading function and questioning what they “even do,” the statement portrays them as incapable, unskilled, and lacking agency or intelligence. The insult is categorical and absolute, leaving no room for individual variation, which aligns with a high incompetence attribution rather than a mild or ambiguous one.

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).\n
Rate the apparent level of competence that the sentence's author seems to express towards the specified target.\n
Choose one label from these seven labels:
high incompetence\n
moderate incompetence\n
slight incompetence\n
neutral, not applicable, not expressed\n
slight competence\n
moderate competence\n
high competence\n
Adhere to the literal meaning of competence, which may be "positive" (e.g., a CEO) or "negative" (e.g., a villain or a dictator). Both types are considered competence, regardless of the outcomes.\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess competence, try to answer the following questions:\n
1. What is the degree of competence towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} in control or out of control? Active or passive? Powerful or weak?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of the competence towards {} only.
You should provide your label in a JSON format whose key is called "label" and that label only. Do not provide your rationale.
"""

### Coarse-grained

In [ ]:
user_role_comp_fs_coarse = """
Some context:\n
Social psychology research has shown that individuals rapidly and subconsciously evaluate others, groups, and even themselves along the dimensions of warmth (W) and competence (C).\n
Your task is to assess the degree of competence towards a specific target within a sentence.\n
Competence, in the broader sense, is interchangeable with 'ability' or 'capability'.
Competence can be defined as ability, power, dominance, being in control, importance, having influence, and assertiveness.\n
Words associated with high competence: hitman, heroical, entrepreneurship, strategies, superman, viper, impunity.\n
Incompetence can be defined as submissiveness, not being in control of a situation, being controlled or guided by outside factors, or weakness.\n
Words associated with low competence: bootlicker, talentless, crash landing, bedridden, underfed.\n

Here are five examples, each containing the text, the target, the score, and the rationale behind the score.\n

Example 1:\n
Target: Women\n
Text: “My wife is the most caring person I've ever met ... she's the only woman in a house full of testosterone . She never stops working whether it's at home or being an RN . I cant keep up but I try . She makes me a better person . I'd be lost without her . Oh and she's smoking hot too.”\n
Competence score: positive competence.\n
Rationale: The author says that his wife “never stops working” in the context of her role as an RN (registered nurse) and as a mother, which suggests multitasking, capability, and professional competence. “I can't keep up but I try” indicates a highly active and energetic individual.\n

Example 2:\n
Target: Hillary Clinton \n
Text: “Would you wanna be in a long term relationship with some bitch that hides her emails, & lies to your face? Then Dont vote \n
Competence score: positive competence.\n
Rationale: Despite the author implying that Clinton is manipulative and dishonest, the author’s phrasing doesn't suggest that she's powerless or ineffective, as the negative behaviors described (concealment and deception) require some degree of agency and planning. The author suggests that Clinton is being deliberate in her (“negative”) performance and active in her (“negative”) effects, leading to a moderately high score in the competence dimension.

Example 3:\n
Target: Women\n
Text: I need feminism because the United States is one of the only countries that doesn't give paid maternity leave.\n
Competence score: neutral.\n
Rationale: The statement implies that women deserve certain rights/benefits, but it doesn't directly attribute dominance or control to women. The statement doesn't characterize women as active, powerful, passive, or weak. The author is advocating for institutional change (paid maternity leave) rather than making claims about women's power or capabilities. This is a policy statement rather than a personal attribution about women's competence. This should help you distinguish between advocacy/policy statements and personal/characteristic attributions.

Example 4:\n
Target: Religious people\n
Text: Could all those who believe in a god please leave. The meeting will now continue for the grown ups only.\n
Competence score: negative competence. \n
Rationale: Religious people are portrayed as needing to be excluded from decision-making processes, suggesting they lack the authority or standing or cognitive abilities to participate in important discussions. Additionally, the "grown ups only" framing explicitly characterizes religious people as childlike, passive, and intellectually weak (i.e., subordinates to "grown ups”). This is a direct attack on their mental capacity and maturity, which are core competence attributes.\n

Example 5:\n
Target: Women\n
Text: What the fuck do women even do? I mean seriously they're just useless other than sex. #womensrights #Feminist \n
Competence score: negative competence. \n
Rationale: The text explicitly depicts women as “useless” and valuable only for sex, which strips them of any functional, intellectual, or productive role. This framing directly denies women competence by asserting they contribute nothing meaningful beyond sexual utility. By reducing women to a single, degrading function and questioning what they “even do,” the statement portrays them as incapable, unskilled, and lacking agency or intelligence. The insult is categorical and absolute, leaving no room for individual variation, which aligns with a high incompetence attribution rather than a mild or ambiguous one.

Instructions:\n
Consider the entire meaning of the sentence before attempting to give the relevant scores.\n
You will be given a text snippet and a target (group, entity, or individual).\n
Rate the apparent level of competence that the sentence's author seems to express towards the specified target.\n

Choose one of these three labels:
negative competence (which can include slight, moderate, or high incompetence).
neutral, not expressed, not applicable\n
positive competence (which can include slight, moderate, or high competence).

Adhere to the literal meaning of competence, which may be "positive" (e.g., a CEO) or "negative" (e.g., a villain or a dictator). Both types are considered competence, regardless of the outcomes.\n

Here is the sentence: {}.\n
Its target is: {}.

In order to assess competence, try to answer the following questions:\n
1. What is the degree of competence towards {} that the author of the text seems to express? \n
2. Does the author seem to perceive {} in control or out of control? Active or passive? Powerful or weak?\n
Remember: even if the speaker is explicitly targeting someone else, since the target is {}, your score should be an assessment of the competence towards {} only.
You should analyse the meaning, then, based on your analysis, add your rating to a key called "label".
You should provide your label in a JSON object whose key is called "label" and that label only.
"""

# **Custom Prompting Functions**

## [Post-processing] Cleaning/Unifying LLM Responses

In [ ]:
def clean_response(response: str) -> dict:
    """
    Takes Gemma's raw JSON-like response string and converts it into
    a valid Python dictionary.
    """
    # Remove code block markers if present
    cleaned = response.strip()
    if cleaned.startswith("```"):
        # slice off the triple backticks and optional 'json'
        cleaned = "\n".join(line for line in cleaned.splitlines() if not line.strip().startswith("```"))

    cleaned = re.sub(r'\\([^"\\/bfnrtu])', r'\\\\\1', cleaned)

    return json.loads(cleaned)

In [ ]:
def clean_response_qwen(response: str) -> dict:
    """
    Takes raw JSON-like response string and converts it into
    a valid Python dictionary.
    """
    # Remove code block markers if present
    cleaned = response.strip()
    if cleaned.startswith("```"):
        # slice off the triple backticks and optional 'json'
        cleaned = "\n".join(line for line in cleaned.splitlines() if not line.strip().startswith("```"))

    cleaned = re.sub(r'\\([^"\\/bfnrtu])', r'\\\\\1', cleaned)

    cleaned = re.sub(
        r'"reason":\s*"(.+?)"\s+([^,]+),\s*"label"',
        lambda m: f'"reason": "{m.group(1)} {m.group(2).strip()}", "label"',
        cleaned,
        flags=re.DOTALL
    )

    return json.loads(cleaned)

In [ ]:
def return_value(object):
  if isinstance(object, dict):
    value = object['label']
    return value if value != None else object
  elif isinstance(object, str):
    try:
      return str(eval(object)['label'])
    except:
      return object
  else:
    return object

In [ ]:
# Fix non-adherance to labels

to_replace = {'low competence': 'high incompetence',
              'low trust': 'high distrust',
              'low sociability': 'high unsociability',
              'not expressed, not applicable, etc.':'neutral',
              'not applicable, not expressed, etc.':'neutral',
              'not expressed':'neutral',
                'neutral, not applicable, not expressed, etc.':'neutral',
                'neutral, not applicable, not expressed':'neutral',
              'high_distrust': 'high distrust',
              'low_trust': 'high distrust',
              'high_trust': 'high trust'}

In [ ]:
convert_dict = {
    'moderate competence': 2,
    'moderate trust': 2,
    'moderate sociability': 2,
    'high competence': 3,
    'high trust': 3,
    'high sociability': 3,
    'slight competence':1,
    'slight trust':1,
    'slight sociability':1,
    'moderate incompetence':-2,
    'moderate distrust':-2,
    'moderate unsociability':-2,
    'high incompetence':-3,
    'high distrust':-3,
    'high unsociability':-3,
    'neutral':0,
    'neutral, not applicable, not expressed, etc.':0,
    'neutral, not applicable, not expressed, etc':0,
    'neutral, not applicable, not expressed':0,
    'slight incompetence':-1,
    'slight unsociability':-1,
    'slight distrust':-1,
    -3: -3,
    -2: -2,
    -1: -1,
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    '+3':3,
    '+2':2,
    '+1':1,
    '-3': -3,
    '-2': -2,
    '-1': -1,
    '0':0
}

## Prompting Gemma and Qwen

In [ ]:
def prompt(pipe, system_message, user_role, target, sentence):
  """
  Use this function to prompt Gemma and Qwen models.
  Generate a single model response using a chat-style prompting setup.

  This function formats a system message and a user prompt, sends them to
  a HuggingFace text-generation pipeline (e.g. Gemma or Qwen), and returns
  the generated assistant content. The user prompt is created by formatting
  `user_role` with the provided sentence and target.

  Parameters
  ----------
  pipe : transformers.Pipeline
      A HuggingFace text-generation pipeline configured for chat-style input.
  system_message : str
      Instruction given to the model describing its role or behavior.
  user_role : str
      A prompt template string that will be formatted with the sentence
      and target (expected to contain multiple `{}` placeholders).
  target : str
      The entity or concept the generation should focus on.
  sentence : str
      The input sentence to condition the generation.

  Returns
  -------
  str
      The generated text produced by the model.
  """
  messages = [
      {"role": "system", "content": system_message},
      {"role": "user", "content": user_role.format(sentence, target, target, target, target, target)}
  ]

  outputs = pipe(
      messages,
      max_new_tokens=800,
      pad_token_id = pipe.tokenizer.eos_token_id )

  return outputs[0]['generated_text'][2]['content']

In [ ]:
def generate_prompts(pipe, df, system_message, user_role_trust, user_role_soc, user_role_comp):
  """
    Prompt Gemma or Qwen for each row in W&C-Sent.

    For every row in the input dataframe, this function generates three
    separate responses (Trust, Sociability, and Competence) by repeatedly
    calling the `prompt` function with different user prompt templates.
    The results are stored in a nested dictionary indexed by row number.

    Parameters
    ----------
    pipe : transformers.Pipeline
        A HuggingFace text-generation pipeline used for all generations.
    df : pandas.DataFrame
        Input dataframe where column 0 contains the target and
        column 1 contains the sentence to summarize.
    system_message : str
        System-level instruction shared across all prompts.
    user_role_trust : str
        Prompt template for generating trust-related summaries.
    user_role_soc : str
        Prompt template for generating sociability-related summaries.
    user_role_comp : str
        Prompt template for generating competence-related summaries.

    Returns
    -------
    dict
        A dictionary where each key corresponds to a row index and each value
        is a dictionary containing the target, sentence, and generated
        summaries for Trust, Sociability, and Competence.
    """

  gen_dict = {}

  for i in tqdm(range(len(df))):

    gen_dict[i] = {}
    gen_dict[i]['Target'] = df.iloc[i, 0]
    gen_dict[i]['Sentence'] = df.iloc[i, 1]
    gen_dict[i]['Trust'] = prompt_1(pipe=pipe,
                                    system_message=system_message,
                                    user_role=user_role_trust,
                                    target=df.iloc[i, 0],
                                    sentence=df.iloc[i, 1])
    gen_dict[i]['Sociability'] = prompt(pipe=pipe,
                                        system_message=system_message,
                                        user_role=user_role_soc,
                                        target=df.iloc[i, 0],
                                        sentence=df.iloc[i, 1])
    gen_dict[i]['Competence'] = prompt(pipe=pipe,
                                        system_message=system_message,
                                        user_role=user_role_comp,
                                        target=df.iloc[i, 0],
                                        sentence=df.iloc[i, 1])
  return gen_dict

## Prompting the OpenAI Models

In [ ]:
def prepare_single_prompt(user_role, target, sentence, extra_payload=None):
      """
      Build chat messages (system + user) by formatting
      `user_role` with `sentence`/`target`, optionally appending batch payload.
      """

    user_content = user_role.format(sentence, target, target, target, target, target)

    if extra_payload is not None:
        user_content += "\n\nBATCH DATA:\n" + extra_payload

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_content}
    ]

    return messages

In [ ]:
def prepare_batch_prompt(batch_df, user_role):
      """Create a single batched user message from a dataframe chunk
      (with per-row IDs) requesting JSON `results` for mapping outputs
      back to rows.
      """
    blocks = []
    for idx, row in batch_df.iterrows():
        user_prompt = user_role.format(
            row["Text"],
            row["Target"],
            row["Target"],
            row["Target"],
            row["Target"],
            row["Target"],
        )
        # Wrap with an ID to map the answer back
        block = f"ID: {idx}\n{user_prompt}"
        blocks.append(block)

    combined_user_content = (
        "You will be given multiple items.\n"
        "For each item, identified by `ID: <number>`, perform the task described.\n"
        "Return a single JSON object with a key `results`, where `results` is a list of\n"
        "objects of the form {\"id\": <ID>, \"score\": <your_output>}.\n\n"
        + "\n\n".join(blocks)
    )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": combined_user_content},
    ]
    return messages

In [ ]:
def run_batch_prompting(dim_df, user_role, batch_size=23):
      """
      Run batched Chat Completions over `dim_df`, parse JSON
      (or fall back to regex/raw text), and return a
      dict of per-ID responses with text/target.
      """
  results_dict = {}

  n = len(dim_df)
  num_batches = math.ceil(n / batch_size)

  for b in tqdm(range(num_batches)):
      batch_df = dim_df.iloc[b * batch_size : (b + 1) * batch_size]

      messages = build_batch_messages(batch_df, user_role)

      completion = client.chat.completions.create(
          model="gpt-4o",
          messages=messages,
          response_format={"type": "json_object"},
          temperature=0,
      )

      content = completion.choices[0].message.content
      parsed_any = False

      # ---------- 1) Try to parse as JSON as intended ----------
      try:
          data = json.loads(content)
      except json.JSONDecodeError:
          data = None

      if isinstance(data, dict):
          results_list = data.get("results", [])
          if isinstance(results_list, list):
              for item in results_list:
                  if not isinstance(item, dict):
                      print(f"[WARN] Batch {b}: item not dict: {item}")
                      continue

                  _id = item.get("id")
                  if _id is None:
                      print(f"[WARN] Batch {b}: missing `id` in item: {item}")
                      continue

                  # Normalize key type (pick ONE and stick to it)
                  try:
                      _id = int(_id)
                  except (TypeError, ValueError):
                      print(f"[WARN] Batch {b}: non-int id: {_id}")
                      continue

                  # Find "score-like" payload
                  score = item.get("score")
                  if score is None:
                      for alt_key in ["label", "value", "rating", "prediction"]:
                          if alt_key in item:
                              score = item[alt_key]
                              break

                  if score is None:
                      # If the model returned other fields (e.g., reason/label) but not "score",
                      # store the whole item so you don't lose anything.
                      score = item

                  # Pull exact sentence/target from THIS batch (not the full df)
                  if _id in batch_df.index:
                      text = batch_df.loc[_id, "Text"]
                      target = batch_df.loc[_id, "Target"]
                  else:
                      # If model returned something unexpected, fall back to global df
                      text = dim_df.loc[_id, "Text"]
                      target = dim_df.loc[_id, "Target"]

                  # Store a rich record instead of a floating "score"
                  results_dict[_id] = {
                      "id": _id,
                      "text": text,
                      "target": target,
                      "response": score,   # rename to "score" if you prefer
                  }

                  parsed_any = True

      # ---------- 2) Fallback: regex over plain text ----------
      if not parsed_any:
          pattern = r"ID:\s*(\d+)[^\d\-]*(-?\d+(?:\.\d+)?)"
          for match in re.finditer(pattern, content):
              _id = int(match.group(1))
              score = match.group(2)

              # Only attach if that ID is actually in this batch
              if _id in batch_df.index:
                  results_dict[_id] = {
                      "id": _id,
                      "text": batch_df.loc[_id, "Text"],
                      "target": batch_df.loc[_id, "Target"],
                      "response": score,
                  }
                  parsed_any = True

          if not parsed_any:
              print(f"[WARN] Batch {b}: could not parse structured scores from content.")

      # ---------- 3) Last-resort fallback: don't lose the batch ----------
      if not parsed_any:
          for _id in batch_df.index:
              _id = int(_id)
              results_dict.setdefault(_id, {
                  "id": _id,
                  "text": batch_df.loc[_id, "Text"],
                  "target": batch_df.loc[_id, "Target"],
                  "response": content,
              })

      if b == 0:
        print(content)

  return results_dict